# W2C2 Lab: Sentiment Showdown

Run every cell from the top. **Everything already works.**

Today you will:

1. Train a working sentiment classifier in five lines of scikit-learn.
2. Find the specific thing it gets wrong, and fix it.
3. Open it up and read the words it thinks are positive.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup. Builds a small movie-review dataset. No download needed.
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

random.seed(0)

GOOD = ["brilliant", "great", "moving", "superb", "wonderful",
        "clever", "funny", "beautiful", "excellent", "inspired"]
BAD = ["dull", "boring", "awful", "weak", "terrible",
       "lazy", "stupid", "lifeless", "poor", "wooden"]
NOUNS = ["film", "script", "cast", "acting", "story",
         "ending", "plot", "direction", "picture", "dialogue"]

PLAIN = ["a {a} {n}", "the {n} was {a}", "a {a} {n} and a {a2} {n2}", "what a {a} {n}"]
# A quarter of the reviews are NEGATED, so a good word turns up in a bad review
# and the other way round. Real reviews do this constantly.
NEGATED = ["the {n} was not {a}", "not a {a} {n}", "the {n} was hardly {a}"]

texts, labels = [], []
for _ in range(600):
    negated = random.random() < 0.25
    positive = random.random() < 0.5
    words = GOOD if (positive != negated) else BAD
    template = random.choice(NEGATED if negated else PLAIN)
    texts.append(template.format(a=random.choice(words), a2=random.choice(words),
                                 n=random.choice(NOUNS), n2=random.choice(NOUNS)))
    labels.append("pos" if positive else "neg")

reviews = pd.DataFrame({"text": texts, "label": labels})
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=0, stratify=labels)

print(f"{len(texts)} reviews: {len(X_train)} to train on, {len(X_test)} held back")
print()
print(reviews.sample(5, random_state=3).to_string(index=False))

## Part 1. A working classifier, in five lines

`CountVectorizer` turns text into a table of word counts. `MultinomialNB`
is Naive Bayes. Together they are a complete sentiment classifier.

In [ ]:
# GIVEN. Train, predict, score.
vectorizer = CountVectorizer()
model = MultinomialNB().fit(vectorizer.fit_transform(X_train), y_train)

predictions = model.predict(vectorizer.transform(X_test))
accuracy = accuracy_score(y_test, predictions)
print(f"accuracy: {accuracy:.2f}   ({int(round(accuracy * len(y_test)))} of {len(y_test)} correct)")
print(f"features: {len(vectorizer.get_feature_names_out())} single words")

In [ ]:
# GIVEN. Where is it going wrong? Look at the misses.
misses = [(t, g, r) for t, g, r in zip(X_test, predictions, y_test) if g != r]
print(f"{len(misses)} mistakes. The first few:")
for text, guess, truth in misses[:8]:
    print(f"   guessed {guess}, really {truth}:  {text}")
print()
print("Look for a pattern. Almost every mistake contains the word 'not' or 'hardly'.")

In [ ]:
# GIVEN. The confusion matrix, drawn.
cm = confusion_matrix(y_test, predictions, labels=["neg", "pos"])
fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.imshow(cm, cmap="Reds")
ax.set_xticks([0, 1], ["neg", "pos"]); ax.set_yticks([0, 1], ["neg", "pos"])
ax.set_xlabel("predicted"); ax.set_ylabel("actually")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=15)
plt.title("Confusion matrix"); plt.show()

In [ ]:
# ================== YOUR TURN 1 ==================
# The vectorizer only counts SINGLE words, so 'not great' looks exactly
# like 'great' to it. Let it count adjacent PAIRS of words as well.
#
# Hint: CountVectorizer(ngram_range=(1, 2))
#
# Expected: accuracy jumps from about 0.77 to about 0.88, and the feature count
#           goes from 36 to over 300. Pairs are the only way a bag-of-words model
#           can tell 'not great' from 'great'.
# ===============================================
vec = CountVectorizer()          # <-- add ngram_range here

m = MultinomialNB().fit(vec.fit_transform(X_train), y_train)
acc = accuracy_score(y_test, m.predict(vec.transform(X_test)))

print(f"features: {len(vec.get_feature_names_out())}")
print(f"accuracy: {acc:.2f}   (was 0.77 with single words)")
pairs = [f for f in vec.get_feature_names_out() if " " in f]
print(f"of those, {len(pairs)} are word pairs, for example: {pairs[:4]}")

## Part 2. Open the box: what did it actually learn?

The model is not magic. It learned a score for every feature in every
class, and you can read them straight off.

In [ ]:
# GIVEN. The words the model finds most positive and most negative.
feature_names = vectorizer.get_feature_names_out()
neg_scores, pos_scores = model.feature_log_prob_     # one row per class
leaning = pos_scores - neg_scores                    # above zero = leans positive

order = np.argsort(leaning)
print("most NEGATIVE:", [feature_names[i] for i in order[:8]])
print("most POSITIVE:", [feature_names[i] for i in order[-8:]])

top = np.concatenate([order[:8], order[-8:]])
plt.figure(figsize=(8, 3.4))
plt.bar([feature_names[i] for i in top], [leaning[i] for i in top],
        color=["#999"] * 8 + ["#7C2529"] * 8)
plt.xticks(rotation=60, ha="right"); plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("leans negative <-> positive"); plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# Write your own review and see what the model makes of it.
# Try to fool it with a negation, such as: the film was not brilliant
#
# Expected: the single-word model is confidently WRONG on negated reviews: it
#           sees 'brilliant' and says pos. That is the same weakness you fixed in
#           task 1, seen one review at a time.
# ===============================================
MY_REVIEW = "the film was not brilliant"          # <-- change me

counts = vectorizer.transform([MY_REVIEW])
guess = model.predict(counts)[0]
confidence = model.predict_proba(counts)[0].max()
print(f"single-word model says: {guess}  (confidence {confidence:.0%})")

recognised = [w for w in MY_REVIEW.lower().split() if w in set(feature_names)]
print("words it recognised:", recognised)

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   vec = CountVectorizer(ngram_range=(1, 2))
#   (1, 2) means single words AND adjacent pairs, so "not brilliant" becomes its
#   own feature with its own score. A quarter of this dataset is negated, which
#   is exactly why single words top out near 0.77 and pairs reach about 0.88.
#
# YOUR TURN 2
#   Any negated review fools the single-word model. It adds up independent word
#   scores and has no idea that "not" reverses the word after it.
#
# The idea to carry forward:
#   a bag of words has no word order. Everything you add on top of it this
#   semester, from bigrams to attention, is a way of buying some of it back.